In [2]:
"""
=============================================================================
SIMPLE PATIENT MANAGEMENT SYSTEM FOR JUPYTER/COLAB
=============================================================================
Institution : Lagos University Teaching Hospital (LUTH)
              College of Medicine, Idiaraba, Lagos
Programme   : MSc Biomedical Engineering, 2025/2026
-----------------------------------------------------------------------------
No Database Required - Uses in-memory storage (Python lists)
Works in: Google Colab, Jupyter Notebook, JupyterLab
-----------------------------------------------------------------------------
Run in Colab/Jupyter:
    !pip install ipywidgets
    # Then run all cells
=============================================================================
"""

import ipywidgets as widgets
from IPython.display import display, clear_output
from datetime import datetime
import random

# ═══════════════════════════════════════════════════════════════════════════
# IN-MEMORY DATA STORAGE (instead of database)
# ═══════════════════════════════════════════════════════════════════════════

# Sample LUTH patient data
patients = [
    {
        "id": "LUTH-2025-001",
        "name": "Adaobi Okonkwo",
        "age": 39,
        "gender": "Female",
        "phone": "08012345678",
        "blood_group": "O+",
        "allergies": "Penicillin"
    },
    {
        "id": "LUTH-2025-002",
        "name": "Emeka Nwosu",
        "age": 52,
        "gender": "Male",
        "phone": "08098765432",
        "blood_group": "A+",
        "allergies": "None"
    },
    {
        "id": "LUTH-2025-003",
        "name": "Fatima Bello",
        "age": 34,
        "gender": "Female",
        "phone": "07033334444",
        "blood_group": "B+",
        "allergies": "Sulfonamides"
    }
]

appointments = [
    {
        "patient_id": "LUTH-2025-001",
        "patient_name": "Adaobi Okonkwo",
        "doctor": "Dr. Adeyemi",
        "department": "Cardiology",
        "date": "2026-02-20",
        "time": "09:00",
        "reason": "Follow-up consultation"
    },
    {
        "patient_id": "LUTH-2025-002",
        "patient_name": "Emeka Nwosu",
        "doctor": "Dr. Ibrahim",
        "department": "Endocrinology",
        "date": "2026-02-18",
        "time": "14:00",
        "reason": "Diabetes management"
    }
]

prescriptions = [
    {
        "patient_id": "LUTH-2025-001",
        "patient_name": "Adaobi Okonkwo",
        "medication": "Amlodipine 5mg",
        "dosage": "5mg once daily",
        "duration": "30 days",
        "doctor": "Dr. Adeyemi",
        "date": "2026-02-15"
    },
    {
        "patient_id": "LUTH-2025-002",
        "patient_name": "Emeka Nwosu",
        "medication": "Metformin 500mg",
        "dosage": "500mg twice daily",
        "duration": "30 days",
        "doctor": "Dr. Ibrahim",
        "date": "2026-02-10"
    }
]

# ═══════════════════════════════════════════════════════════════════════════
# HELPER FUNCTIONS
# ═══════════════════════════════════════════════════════════════════════════

def generate_patient_id():
    """Generate a new patient ID"""
    return f"LUTH-2025-{len(patients) + 1:03d}"

def find_patient(patient_id):
    """Find a patient by ID"""
    for patient in patients:
        if patient["id"] == patient_id:
            return patient
    return None

# ═══════════════════════════════════════════════════════════════════════════
# TAB 1: REGISTER NEW PATIENT
# ═══════════════════════════════════════════════════════════════════════════

def create_register_tab():
    """Create patient registration interface"""
    
    # Input fields
    name_input = widgets.Text(
        description='Full Name:',
        placeholder='Enter patient name',
        style={'description_width': '120px'}
    )
    
    age_input = widgets.IntText(
        description='Age:',
        value=30,
        style={'description_width': '120px'}
    )
    
    gender_input = widgets.Dropdown(
        description='Gender:',
        options=['Male', 'Female', 'Other'],
        value='Male',
        style={'description_width': '120px'}
    )
    
    phone_input = widgets.Text(
        description='Phone:',
        placeholder='080XXXXXXXX',
        style={'description_width': '120px'}
    )
    
    blood_input = widgets.Dropdown(
        description='Blood Group:',
        options=['A+', 'A-', 'B+', 'B-', 'AB+', 'AB-', 'O+', 'O-'],
        value='O+',
        style={'description_width': '120px'}
    )
    
    allergy_input = widgets.Text(
        description='Allergies:',
        placeholder='None or list allergies',
        style={'description_width': '120px'}
    )
    
    output = widgets.Output()
    
    # Register button
    def register_patient(b):
        with output:
            clear_output()
            if not name_input.value:
                print("❌ Error: Patient name is required!")
                return
            
            new_patient = {
                "id": generate_patient_id(),
                "name": name_input.value,
                "age": age_input.value,
                "gender": gender_input.value,
                "phone": phone_input.value,
                "blood_group": blood_input.value,
                "allergies": allergy_input.value or "None"
            }
            
            patients.append(new_patient)
            print(f"✅ Patient registered successfully!")
            print(f"   Patient ID: {new_patient['id']}")
            print(f"   Name: {new_patient['name']}")
            
            # Clear form
            name_input.value = ""
            phone_input.value = ""
            allergy_input.value = ""
    
    register_btn = widgets.Button(
        description='Register Patient',
        button_style='success',
        icon='check'
    )
    register_btn.on_click(register_patient)
    
    # Layout
    title = widgets.HTML('<h3>📝 Register New Patient</h3>')
    
    return widgets.VBox([
        title,
        name_input,
        age_input,
        gender_input,
        phone_input,
        blood_input,
        allergy_input,
        register_btn,
        output
    ])

# ═══════════════════════════════════════════════════════════════════════════
# TAB 2: VIEW PATIENT RECORDS
# ═══════════════════════════════════════════════════════════════════════════

def create_records_tab():
    """Create patient records viewer"""
    
    search_input = widgets.Text(
        description='Search:',
        placeholder='Enter patient name or ID',
        style={'description_width': '100px'}
    )
    
    output = widgets.Output()
    
    def show_all_patients(b=None):
        with output:
            clear_output()
            if not patients:
                print("No patients registered yet.")
                return
            
            print("="*80)
            print("🏥 LUTH PATIENT RECORDS")
            print("="*80)
            for p in patients:
                print(f"\nID: {p['id']}")
                print(f"Name: {p['name']}")
                print(f"Age: {p['age']} | Gender: {p['gender']}")
                print(f"Phone: {p['phone']} | Blood: {p['blood_group']}")
                print(f"Allergies: {p['allergies']}")
                print("-"*80)
    
    def search_patients(b):
        with output:
            clear_output()
            query = search_input.value.lower()
            if not query:
                show_all_patients()
                return
            
            found = [p for p in patients 
                    if query in p['name'].lower() or query in p['id'].lower()]
            
            if not found:
                print(f"❌ No patients found matching '{search_input.value}'")
                return
            
            print("="*80)
            print(f"🔍 SEARCH RESULTS ({len(found)} found)")
            print("="*80)
            for p in found:
                print(f"\nID: {p['id']}")
                print(f"Name: {p['name']}")
                print(f"Age: {p['age']} | Gender: {p['gender']}")
                print(f"Phone: {p['phone']} | Blood: {p['blood_group']}")
                print(f"Allergies: {p['allergies']}")
                print("-"*80)
    
    search_btn = widgets.Button(
        description='Search',
        button_style='primary',
        icon='search'
    )
    search_btn.on_click(search_patients)
    
    show_all_btn = widgets.Button(
        description='Show All',
        button_style='info',
        icon='list'
    )
    show_all_btn.on_click(show_all_patients)
    
    # Auto-show all on load
    show_all_patients()
    
    title = widgets.HTML('<h3>🗂 Patient Records</h3>')
    search_box = widgets.HBox([search_input, search_btn, show_all_btn])
    
    return widgets.VBox([title, search_box, output])

# ═══════════════════════════════════════════════════════════════════════════
# TAB 3: APPOINTMENTS
# ═══════════════════════════════════════════════════════════════════════════

def create_appointments_tab():
    """Create appointments interface"""
    
    # Input fields
    patient_id_input = widgets.Text(
        description='Patient ID:',
        placeholder='LUTH-2025-001',
        style={'description_width': '120px'}
    )
    
    doctor_input = widgets.Text(
        description='Doctor:',
        placeholder='Dr. Name',
        style={'description_width': '120px'}
    )
    
    dept_input = widgets.Dropdown(
        description='Department:',
        options=['Cardiology', 'Endocrinology', 'Paediatrics', 'Surgery', 
                'Radiology', 'Laboratory', 'Pharmacy'],
        style={'description_width': '120px'}
    )
    
    date_input = widgets.Text(
        description='Date:',
        value='2026-02-20',
        placeholder='YYYY-MM-DD',
        style={'description_width': '120px'}
    )
    
    time_input = widgets.Text(
        description='Time:',
        value='09:00',
        placeholder='HH:MM',
        style={'description_width': '120px'}
    )
    
    reason_input = widgets.Text(
        description='Reason:',
        placeholder='Consultation reason',
        style={'description_width': '120px'}
    )
    
    output = widgets.Output()
    
    def book_appointment(b):
        with output:
            clear_output()
            patient = find_patient(patient_id_input.value)
            
            if not patient:
                print(f"❌ Error: Patient ID '{patient_id_input.value}' not found!")
                return
            
            new_appt = {
                "patient_id": patient_id_input.value,
                "patient_name": patient['name'],
                "doctor": doctor_input.value,
                "department": dept_input.value,
                "date": date_input.value,
                "time": time_input.value,
                "reason": reason_input.value
            }
            
            appointments.append(new_appt)
            
            print("✅ Appointment booked successfully!")
            print(f"   Patient: {patient['name']}")
            print(f"   Doctor: {doctor_input.value}")
            print(f"   Date: {date_input.value} at {time_input.value}")
            print("\n" + "="*60)
            show_appointments()
    
    def show_appointments(b=None):
        if not appointments:
            print("No appointments scheduled yet.")
            return
        
        print("\n📅 UPCOMING APPOINTMENTS")
        print("="*80)
        for appt in appointments:
            print(f"\nPatient: {appt['patient_name']} ({appt['patient_id']})")
            print(f"Doctor: {appt['doctor']} | Dept: {appt['department']}")
            print(f"Date: {appt['date']} at {appt['time']}")
            print(f"Reason: {appt['reason']}")
            print("-"*80)
    
    book_btn = widgets.Button(
        description='Book Appointment',
        button_style='success',
        icon='calendar'
    )
    book_btn.on_click(book_appointment)
    
    view_btn = widgets.Button(
        description='View All',
        button_style='info',
        icon='list'
    )
    view_btn.on_click(lambda b: (output.clear_output(), show_appointments()))
    
    # Auto-show appointments on load
    with output:
        show_appointments()
    
    title = widgets.HTML('<h3>📅 Appointments</h3>')
    subtitle = widgets.HTML('<h4>Book New Appointment</h4>')
    
    return widgets.VBox([
        title,
        subtitle,
        patient_id_input,
        doctor_input,
        dept_input,
        date_input,
        time_input,
        reason_input,
        widgets.HBox([book_btn, view_btn]),
        output
    ])

# ═══════════════════════════════════════════════════════════════════════════
# TAB 4: PRESCRIPTIONS
# ═══════════════════════════════════════════════════════════════════════════

def create_prescriptions_tab():
    """Create prescriptions interface"""
    
    # Input fields
    patient_id_input = widgets.Text(
        description='Patient ID:',
        placeholder='LUTH-2025-001',
        style={'description_width': '120px'}
    )
    
    medication_input = widgets.Text(
        description='Medication:',
        placeholder='Drug name',
        style={'description_width': '120px'}
    )
    
    dosage_input = widgets.Text(
        description='Dosage:',
        placeholder='e.g., 5mg twice daily',
        style={'description_width': '120px'}
    )
    
    duration_input = widgets.Text(
        description='Duration:',
        placeholder='e.g., 30 days',
        style={'description_width': '120px'}
    )
    
    doctor_input = widgets.Text(
        description='Doctor:',
        placeholder='Dr. Name',
        style={'description_width': '120px'}
    )
    
    output = widgets.Output()
    
    def issue_prescription(b):
        with output:
            clear_output()
            patient = find_patient(patient_id_input.value)
            
            if not patient:
                print(f"❌ Error: Patient ID '{patient_id_input.value}' not found!")
                return
            
            new_rx = {
                "patient_id": patient_id_input.value,
                "patient_name": patient['name'],
                "medication": medication_input.value,
                "dosage": dosage_input.value,
                "duration": duration_input.value,
                "doctor": doctor_input.value,
                "date": datetime.now().strftime("%Y-%m-%d")
            }
            
            prescriptions.append(new_rx)
            
            print("✅ Prescription issued successfully!")
            print(f"   Patient: {patient['name']}")
            print(f"   Medication: {medication_input.value}")
            print(f"   Dosage: {dosage_input.value}")
            print("\n" + "="*60)
            show_prescriptions()
    
    def show_prescriptions(b=None):
        if not prescriptions:
            print("No prescriptions issued yet.")
            return
        
        print("\n💊 PRESCRIPTION HISTORY")
        print("="*80)
        for rx in prescriptions:
            print(f"\nPatient: {rx['patient_name']} ({rx['patient_id']})")
            print(f"Medication: {rx['medication']}")
            print(f"Dosage: {rx['dosage']}")
            print(f"Duration: {rx['duration']}")
            print(f"Doctor: {rx['doctor']} | Date: {rx['date']}")
            print("-"*80)
    
    issue_btn = widgets.Button(
        description='Issue Prescription',
        button_style='success',
        icon='pills'
    )
    issue_btn.on_click(issue_prescription)
    
    view_btn = widgets.Button(
        description='View All',
        button_style='info',
        icon='list'
    )
    view_btn.on_click(lambda b: (output.clear_output(), show_prescriptions()))
    
    # Auto-show prescriptions on load
    with output:
        show_prescriptions()
    
    title = widgets.HTML('<h3>💊 Prescriptions</h3>')
    subtitle = widgets.HTML('<h4>Issue New Prescription</h4>')
    
    return widgets.VBox([
        title,
        subtitle,
        patient_id_input,
        medication_input,
        dosage_input,
        duration_input,
        doctor_input,
        widgets.HBox([issue_btn, view_btn]),
        output
    ])

# ═══════════════════════════════════════════════════════════════════════════
# MAIN APPLICATION
# ═══════════════════════════════════════════════════════════════════════════

def create_app():
    """Create the main application with tabs"""
    
    # Header
    header = widgets.HTML("""
        <div style='background-color: #003366; padding: 15px; border-radius: 5px; margin-bottom: 10px;'>
            <h2 style='color: white; text-align: center; margin: 0;'>
                🏥 LAGOS UNIVERSITY TEACHING HOSPITAL (LUTH)
            </h2>
            <p style='color: #AACCFF; text-align: center; margin: 5px 0 0 0;'>
                College of Medicine, Idiaraba | Patient Management System
            </p>
        </div>
    """)
    
    # Create tabs
    tab = widgets.Tab()
    tab.children = [
        create_register_tab(),
        create_records_tab(),
        create_appointments_tab(),
        create_prescriptions_tab()
    ]
    
    tab.set_title(0, '📝 Register Patient')
    tab.set_title(1, '🗂 Patient Records')
    tab.set_title(2, '📅 Appointments')
    tab.set_title(3, '💊 Prescriptions')
    
    # Footer
    footer = widgets.HTML("""
        <div style='text-align: center; margin-top: 10px; color: #666; font-size: 11px;'>
            MSc Biomedical Engineering 2025/2026 | University of Lagos
        </div>
    """)
    
    return widgets.VBox([header, tab, footer])

# ═══════════════════════════════════════════════════════════════════════════
# RUN THE APPLICATION
# ═══════════════════════════════════════════════════════════════════════════

# Display the app
app = create_app()
display(app)

print("✅ LUTH Patient Management System loaded successfully!")
print("   Use the tabs above to manage patients, appointments, and prescriptions.")
print("   All data is stored in memory (no database required).")

✅ LUTH Patient Management System loaded successfully!
   Use the tabs above to manage patients, appointments, and prescriptions.
   All data is stored in memory (no database required).
